In [1]:
!git clone https://github.com/bhujbalatharva252-sketch/Recommendation-Systems-benchmarking

Cloning into 'Recommendation-Systems-benchmarking'...
remote: Enumerating objects: 70, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 70 (delta 4), reused 10 (delta 1), pack-reused 56 (from 1)
Receiving objects: 100% (70/70), 43.00 MiB | 7.33 MiB/s, done.
Resolving deltas: 100% (19/19), done.
Updating files: 100% (35/35), done.


In [2]:
%cd /content/Recommendation-Systems-benchmarking

/content/Recommendation-Systems-benchmarking


In [3]:
import pandas as pd
import numpy as np
import pickle

# Load processed datasets
train = pd.read_csv("data/processed/train.csv")

test = pd.read_csv("data/processed/test.csv")

movies = pd.read_csv("data/raw/movies.dat",
    sep="::",
    engine="python",
    names=["MovieID", "Title", "Genres"],
    encoding="latin-1")

candidate_sets= pd.read_pickle("data/processed/candidate_sets_100.pkl")


# Hit@10

In [4]:

# Hit@10=Checks whether the true test item appears in the top 10 recommended items.

def hit_at_k(ranked_items, test_item, k=10):
    if test_item in ranked_items[:k]:
        return 1.0

    return 0.0

# NDCG@10

In [6]:
# NDCG@10 = Measures ranking quality. Higher score is given when the true item appears higher in the top 10.

def ndcg_at_k(ranked_items, test_item, k=10):
    top_k=ranked_items[:k]

    # Check whether the true item is recommended
    if test_item in top_k:

        # Get position of true item (starting from 1)
        rank=top_k.index(test_item)+1

        # Discount score based on ranking position
        return 1.0/np.log2(rank+1)

    return 0.0


# Evaluate Model

In [7]:
# Evaluate a recommendation model.
    # This function will be used for all recommendation models.
    # It calculates the average Hit@10 and NDCG@10 using the fixed candidate set
def evaluate_model(score_fn, candidate_sets, k=10):
    hits=[]
    ndcgs=[]

    # Evaluate every user
    for user_id,candidate in candidate_sets.items():

        # Get 100 candidate movies
        items= candidate["items"]

        # Actual movie from test set
        test_item= candidate["test_item"]


        # Get model prediction scores
        scores= score_fn(user_id, items)


        # Rank movies according to predicted scores
        ranked_items= [item for item, score in sorted(
                zip(items, scores),
                key=lambda x: x[1],
                reverse=True)]


        # Calculate metrics
        hits.append(hit_at_k(ranked_items, test_item, k))

        ndcgs.append(
            ndcg_at_k(ranked_items, test_item, k)
        )


    # Return average performance over all users
    return {
        "Hit@10": np.mean(hits),
        "NDCG@10": np.mean(ndcgs)
    }

In [12]:
# Example
ranked_items=[10,25,30,40,50]
test_item=10

print("Hit@10:",hit_at_k(ranked_items,test_item,k=10))
print("NDCG@10:",ndcg_at_k(ranked_items,test_item,k=10))

Hit@10: 1.0
NDCG@10: 1.0


In [15]:
# Example 2
ranked_items= [25, 30, 10, 40, 50]
test_item= 10
print("Hit@10:",hit_at_k(ranked_items,test_item,k=10))
print("NDCG@10:",ndcg_at_k(ranked_items,test_item,k=10))

Hit@10: 1.0
NDCG@10: 0.5
